# 02 — Clustering K-Means & Labellisation des communes

Ce notebook applique un clustering **K-Means** sur les communes Allego pour créer les labels supervisés (`0 = Sous-équipé`, `1 = Normalement équipé`, `2 = Bien équipé`).

Ces labels seront utilisés comme variable cible dans le notebook de modélisation.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

ROOT = Path().resolve()
sys.path.insert(0, str(ROOT / "src"))
PLOTS_DIR = ROOT / "plots"

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 120

LABEL_NAMES  = {0: "Sous-équipé", 1: "Normalement équipé", 2: "Bien équipé"}
LABEL_COLORS = {0: "#e74c3c", 1: "#f39c12", 2: "#2ecc71"}


## 1. Chargement & agrégation par commune

In [ ]:
RAW_CSV = ROOT / "data" / "raw" / "consolidation-etalab-schema-irve-statique-v-2.3.1-20260505.csv"
df = pd.read_csv(RAW_CSV, low_memory=False)
df = df[df["nom_operateur"] == "Allego"].copy()
df["latitude"]  = pd.to_numeric(df["consolidated_latitude"],  errors="coerce")
df["longitude"] = pd.to_numeric(df["consolidated_longitude"], errors="coerce")
df = df.dropna(subset=["latitude", "longitude"])

commune_df = df.groupby("consolidated_commune").agg(
    nb_pdc     = ("id_pdc_itinerance", "count"),
    latitude   = ("latitude",  "mean"),
    longitude  = ("longitude", "mean"),
).reset_index()

print(f"Communes analysées : {len(commune_df)}")
print(commune_df["nb_pdc"].describe())


## 2. Distribution du nb de PDC par commune (asymétrie)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(commune_df["nb_pdc"], bins=30, color="#3498db", edgecolor="white")
axes[0].set_title("Distribution de nb_pdc (brute)")
axes[0].set_xlabel("Nb de PDC")

axes[1].hist(np.log1p(commune_df["nb_pdc"]), bins=30, color="#e74c3c", edgecolor="white")
axes[1].set_title("Distribution de log(nb_pdc + 1)")
axes[1].set_xlabel("log(Nb de PDC + 1)")

plt.suptitle("Asymétrie de la distribution — justification du log", y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_distribution_nb_pdc.png", bbox_inches="tight")
plt.show()


## 3. Courbe du coude (Elbow Method) — choix de k

In [ ]:
X_log = np.log1p(commune_df[["nb_pdc"]].values)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

inertias = []
sil_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, inertias, "o-", color="#3498db", linewidth=2)
axes[0].axvline(x=3, color="#e74c3c", linestyle="--", label="k=3 choisi")
axes[0].set_xlabel("Nombre de clusters k")
axes[0].set_ylabel("Inertie (within-cluster SSE)")
axes[0].set_title("Courbe du coude (Elbow Method)")
axes[0].legend()

axes[1].plot(K_range, sil_scores, "s-", color="#2ecc71", linewidth=2)
axes[1].axvline(x=3, color="#e74c3c", linestyle="--", label="k=3 choisi")
axes[1].set_xlabel("Nombre de clusters k")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Score de silhouette par k")
axes[1].legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / "07_elbow_silhouette.png")
plt.show()

print(f"Silhouette score pour k=3 : {sil_scores[1]:.3f}")


## 4. Application du K-Means (k=3)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
raw_labels = kmeans.fit_predict(X_scaled)

cluster_means = pd.Series(commune_df["nb_pdc"].values).groupby(raw_labels).mean()
rank_map = {c: rank for rank, c in enumerate(cluster_means.sort_values().index)}
commune_df["label"] = [rank_map[c] for c in raw_labels]

for label_id, name in LABEL_NAMES.items():
    subset = commune_df[commune_df["label"] == label_id]
    print(f"  Cluster {label_id} — {name:25s} | {len(subset):3d} communes | "
          f"moy. {subset['nb_pdc'].mean():.1f} PDC | "
          f"max {subset['nb_pdc'].max()} PDC")


## 5. Carte des communes par cluster

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))

for label_id, name in LABEL_NAMES.items():
    subset = commune_df[commune_df["label"] == label_id]
    ax.scatter(
        subset["longitude"], subset["latitude"],
        label=name, color=LABEL_COLORS[label_id],
        alpha=0.8, s=40, edgecolors="white", linewidths=0.3,
    )

ax.set_xlim(-5.5, 10.5)
ax.set_ylim(41, 52)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Communes Allego — Niveaux d'équipement (K-Means k=3)")
ax.legend(title="Niveau", framealpha=0.9)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "08_carte_clusters.png")
plt.show()


---
**Fin du notebook 02.** → Lancer `scripts/train.py` puis passer au notebook `03_modelisation.ipynb`